In [1]:
from glob import glob
import pandas as pd
from tqdm import tqdm

from locations import extract_url_map
from events import parse_mmd_taxonomy, extract_events
from timespan import parse_timespan

In [2]:
root_path = "../data/schede mappatura/"

skip = {
    # "david_ruth_FEGB_E_00007": {"rows": 6, "cols": 1}
    "stern_IS_S_00142": {"rows": 2, "cols": 0},
}

In [3]:
chrono_schede = glob(f"{root_path}*/chronotop*")
chrono_schede

['../data/schede mappatura/bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx',
 '../data/schede mappatura/david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx',
 '../data/schede mappatura/bruenn_ charlotte_IS_S_00027/chronotopoi_charlotte_bruenn_IS_S_00027 (bozza).xlsx',
 '../data/schede mappatura/0_template/chronotopoi_template.xlsx',
 '../data/schede mappatura/stern_IS_S_00142/chronotopoi_josef_stern_familie_v3.xlsx',
 '../data/schede mappatura/stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx']

## A list of individual sources for experimentation

ignored in the oveall logic

In [4]:
# current = "bruenn_jehoshua_IS_S_00028/chronotopoi_bruenn_jehoshua_bozza.xlsx"
# current = "david_ruth_FEGB_E_00007/chronotopi_ruth_david_FEGB_E_00007.xlsx"
# current = "bruenn_ charlotte_IS_S_00027/chronotopi_charlotte_bruenn_IS_S_00027 (file revisionato).xlsx"
# current = "stern_IS_S_00142/chronotopi_josef_stern_IS_S_00142.xlsx"
current = "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx"

chrono_schede = [s for s in chrono_schede if current in s]
chrono_schede

['../data/schede mappatura/stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx']

After identification of all sources

# Shortlist processable sources

In [5]:
overview = {
    "stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx": [
        "Josef Stern",
    ],
}

## Put all data together

In [6]:
first = True
df_list = []
for k, v in overview.items():
    for s in v:
        name = k.split("/")[0]
        # print(k,s)
        if name in skip:
            df = pd.read_excel(
                root_path + k, sheet_name=s, skiprows=skip[name]["rows"]
            ).iloc[:, skip[name]["cols"] :]
        else:
            df = pd.read_excel(root_path + k, sheet_name=s)

        print(k, s, df.columns)
        df.columns = [
            "event_label",
            "event_type",
            "place_name",
            "place_type",
            "place_category",
            "wikidata_qid",
            "geonames_id",
            "google maps",
            "date_certainty",
            "date_label",
            "memorial_inscription",
            "source_doc",
            "source_timecode",
            "source_quote",
            "external_links",
            "notes",
        ]

        # merge columns 6+ to notes
        df["notes"] = df[["notes"] + list(df.columns[6:])].apply(
            lambda row: " ".join(row.dropna().astype(str)), axis=1
        )
        # df = df.drop(df.columns[6:], axis=1)

        df["protagonist"] = name
        df["name"] = s
        df_list += [df]
df = pd.concat(df_list, axis=0).astype(str)
df.fillna("", inplace=True)

# Track start/end locations: end_location = current row's place,
# start_location = previous event's place (per person)
start_locations = []
prev_location = {}  # protagonist → last place_name
for _, row in df.iterrows():
    person = row["protagonist"]
    end_loc = row["place_name"].strip()
    start_loc = prev_location.get(person, end_loc)  # fallback to same as end
    start_locations.append(start_loc)
    if end_loc:
        prev_location[person] = end_loc
df["start_location"] = start_locations
df["end_location"] = df["place_name"]

# Collect concepts from event_type, place_type, place_category
concept_labels = set()
for col in ["event_type", "place_type", "place_category"]:
    for val in df[col]:
        v = val.strip()
        if v and v != "nan":
            concept_labels.add(v)
print(f"Concepts to create: {sorted(concept_labels)}")

df


stern_IS_S_00142/chronotopoi_josef_stern_familie_v4.xlsx Josef Stern Index(['event_label', 'event_type', 'place_name', 'place_type',
       'place_category', 'wikidata_qid', 'geonames_id', 'google maps',
       'date_certainty', 'date_label', 'memorial_inscription', 'source_doc',
       'source_timecode', 'source_quote', 'external_links', 'notes'],
      dtype='str')


,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,memorial_inscription,source_doc,source_timecode,source_quote,external_links,notes,protagonist,name
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,,,,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,,,,,,probable 1928-1932,stern_IS_S_00142,Josef Stern
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,,,,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,,,29min 04,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,,,30min 57s,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,,,31min 31s,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern


# Locations

In [7]:
# locs = {n:l for l, n in df["location"].apply(lambda x: extract_urls(x)).to_list()}
locs = {}
for row in tqdm(df.to_dict(orient="records")):
    # print(row)
    # print(extract_urls(row))
    name = row["place_name"]
    locs[name] = {}
    place_labels = set()
    if "place_type" in row and row["place_type"].strip():
        place_labels |= {row["place_type"].strip()}
    if row["place_category"].strip():
        place_labels |= {row["place_category"].strip()}
    locs[name]["label"] = ",".join(place_labels)

    urls = extract_url_map(row["external_links"])
    if (
        "www.wikidata.org" not in urls
        and "wikidata_id" in row
        and row["wikidata_qid"].strip()
    ):
        locs[name]["www.wikidata.org"] = (
            "https://www.wikidata.org/wiki/" + row["wikidata_qid"].strip()
        )
    if (
        "www.geonames.org" not in urls
        and "geonames_id" in row
        and row["geonames_id"].strip()
    ):
        locs[name]["www.geonames.org"] = (
            "https://www.geonames.org/" + row["geonames_id"].strip().removesuffix(".0")
        )

    locs[name].update(urls[0])
print(locs)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 33/33 [00:00<00:00, 16815.94it/s]

{'Gießen, Marktplatz 11': {'label': 'ort_der_zeit', 'www.geonames.org': 'https://www.geonames.org/2920512.0'}, 'Gießen': {'label': 'reise_zurueck', 'www.geonames.org': 'https://www.geonames.org/2920512/giessen.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q564579'}, 'Löberstraße 20': {'label': 'alte_heimat', 'www.giessen.de': 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1'}, 'Ghettohaus, Walltorstraße 48': {'label': 'alte_heimat,Ghettohaus', 'www.giessen.de': 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1'}, 'Schlesien': {'label': 'alte_heimat'}, 'Berlin': {'label': 'alte_heimat', 'www.geonames.org': 'https://www.geonames.org/2950159/berlin.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q64'}, 'Synagoge am Börneplatz, Frankfurt/M': {'label': 'alte_heimat,Synagoge', 'www.geonames.org': 'https://www.geonames.org/6553153/frankfurt-am-main.html', 'www.wikidata.org': 'https://www.wikidata.org/wiki/Q

## Update preexisting locations

In [8]:
import os
import re

def _normalize_loc(name):
    """Normalize for matching: lowercase, no punctuation, sorted words."""
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

if os.path.exists("locations.xlsx"):
    rich = pd.read_excel("locations.xlsx", dtype=str)
    rich = rich.set_index(["location"])
else:
    rich = pd.DataFrame()
    rich.index.name = "location"

# Build a normalized index for flexible matching
existing_keys = {_normalize_loc(idx): idx for idx in rich.index}

for location, row in locs.items():
    key = _normalize_loc(location)

    if key in existing_keys:
        # Enrich existing row: only fill absent cells
        real_idx = existing_keys[key]
        if isinstance(row, dict):
            for col, val in row.items():
                if col not in rich.columns:
                    rich[col] = ""
                existing = rich.loc[real_idx, col]
                if isinstance(existing, pd.Series):
                    existing = existing.iloc[0]
                if pd.isna(existing) or str(existing).strip() in ("", "nan"):
                    rich.loc[real_idx, col] = str(val)
    else:
        # New location: add row with provided values
        if isinstance(row, dict):
            for col in row:
                if col not in rich.columns:
                    rich[col] = ""
            rich.loc[location] = {col: str(val) for col, val in row.items()}
        else:
            rich.loc[location] = pd.Series(dtype=str)
        existing_keys[key] = location

rich.to_excel("locations.xlsx")

# Timespan

In [11]:
df[["time_start", "time_end"]] = (
    df["date_label"].apply(lambda ts: list(parse_timespan(ts).as_tuple())).tolist()
)
df

,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,...,source_doc,source_timecode,source_quote,external_links,notes,protagonist,name,event,time_start,time_end
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,...,,,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern,"[alte Heimat, Geburt]",1921-06-15,1921-06-15
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,...,,,,,probable 1928-1932,stern_IS_S_00142,Josef Stern,"[alte Heimat, Grundschule]",1928-01-01,1932-12-31
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,...,,,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern,"[Realgymnasium, alte Heimat]",1932-01-01,1932-12-31
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,...,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern,[Wohnort],1933-01-01,1933-12-31
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,...,,29min 04,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern,[Abgang von der Schule],1935-01-01,1935-12-31
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,...,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern,[Wohnort],1935-01-01,1935-12-31
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,...,,30min 57s,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern,[Hachschara],1935-01-01,1935-12-31
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,...,,31min 31s,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern,"[Verwandte, Bei n]",1935-01-01,1935-12-31
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,...,,,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern,[Jeschiwa],1936-01-01,1936-12-31
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,...,,,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern,"[Abschied von den Eltern, Abfahrt, >]",1936-01-01,1936-12-31


# Notes

left unprocessed for now

In [12]:
set(df["notes"])

{'11974166.0 https://www.wikidata.org/wiki/Q116016915\nhttps://www.geonames.org/11974166/grossen-linden.html',
 '2888549.0 https://www.wikidata.org/wiki/Q1571834\nhttps://www.geonames.org/2888549/klein-linden.html',
 '2891951.0 probable 1936 https://www.wikidata.org/wiki/Q15979\nhttps://www.geonames.org/2891951/kehl.html',
 '2920512.0 certain 15/06/1921 https://www.geonames.org/2920512/giessen.html\nhttps://www.wikidata.org/wiki/Q564579\nhttps://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1',
 '2920512.0 probable 1975 https://www.geonames.org/2920512/giessen.html\nhttps://www.wikidata.org/wiki/Q564579',
 '293304.0 uncertain 1940-1944 https://www.wikidata.org/wiki/Q550025\nhttps://www.geonames.org/293304/tirat-tsvi.html',
 '2950159.0 probable 1935 31min 31s https://www.wikidata.org/wiki/Q64\nhttps://www.geonames.org/2950159/berlin.html',
 '2995469.0 probable 1936 https://www.wikidata.org/wiki/Q23482\nhttps://www.geonames.org/2995469/marseille.html',
 '31min 58

# Links

left unprocessed for now

In [16]:
urls = set()
for cell in df["external_links"]:
    if pd.notna(cell):
        for url in str(cell).split("\n"):
            url = url.strip()
            if url:
                urls.add(url)
urls

{'https://www.geonames.org/11974166/grossen-linden.html',
 'https://www.geonames.org/2888549/klein-linden.html',
 'https://www.geonames.org/2891951/kehl.html',
 'https://www.geonames.org/2920512/giessen.html',
 'https://www.geonames.org/293165/jezreel-valley.html',
 'https://www.geonames.org/293304/tirat-tsvi.html',
 'https://www.geonames.org/294801/haifa.html',
 'https://www.geonames.org/2950159/berlin.html',
 'https://www.geonames.org/295211/-en-hanaziv.html',
 'https://www.geonames.org/2995469/marseille.html',
 'https://www.geonames.org/6290300/frankfurt-hauptbahnhof.html',
 'https://www.geonames.org/6553153/frankfurt-am-main.html',
 'https://www.giessen.de/index.php?ModID=7&FID=2874.2939.1&object=tx%7C2874.2939.1',
 'https://www.wikidata.org/wiki/Q111620976',
 'https://www.wikidata.org/wiki/Q116016915',
 'https://www.wikidata.org/wiki/Q1375288',
 'https://www.wikidata.org/wiki/Q1571834',
 'https://www.wikidata.org/wiki/Q15979',
 'https://www.wikidata.org/wiki/Q165368',
 'https://ww

# Events

TODO: incomplete due to too much noise. Issues:

- use LL or Lebenslauf, currently extracted as one, but need to be two equivalent
- "alter heimant" instread of "alte heimat"
- "Transport", "Tod des Vaters",  are not label

In [9]:
event_taxonomy = parse_mmd_taxonomy("../docs/tassonomia.mmd")
events = sorted(set(event_taxonomy.keys()), key=lambda x: -len(x))
len(events), events[:5] + ["..."] + events[-5:]

(340,
 ['Città/regioni tedesche, austriache, ceche',
  'SR – Spazi sociali / istituzioni',
  'Friedrichswerdersche Gymnasium',
  'Europa – altri paesi e luoghi',
  'Berlino – quartieri e luoghi',
  '...',
  'Zug',
  'SPD',
  'USA',
  'KPD',
  'Tod'])

In [10]:
df["event"] = df["event_label"].apply(lambda x: extract_events(x, events))
df

,event_label,event_type,place_name,place_type,place_category,wikidata_qid,geonames_id,google maps,date_certainty,date_label,memorial_inscription,source_doc,source_timecode,source_quote,external_links,notes,protagonist,name,event
0,Alte Heimat Geburt,birth,"Gießen, Marktplatz 11",,alte_heimat,Q564579,2920512.0,,certain,15/06/1921,,,,,https://www.geonames.org/2920512/giessen.html\...,2920512.0 certain 15/06/1921 https://www.geona...,stern_IS_S_00142,Josef Stern,"[alte Heimat, Geburt]"
1,Alte Heimat Grundschule,education,Gießen,,alte_heimat,,,,probable,1928-1932,,,,,,probable 1928-1932,stern_IS_S_00142,Josef Stern,"[alte Heimat, Grundschule]"
2,Alte Heimat Realgymnasium,education,Gießen,,alte_heimat,,,,uncertain,1932- Winter 1935,,,,,,uncertain 1932- Winter 1935,stern_IS_S_00142,Josef Stern,"[Realgymnasium, alte Heimat]"
3,Wohnort,residence,Löberstraße 20,,alte_heimat,,,,uncertain,1933?,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,Aufräumung v4: place_name korrigiert von 'Löbe...,stern_IS_S_00142,Josef Stern,[Wohnort]
4,Abgang von der Schule,education,Gießen,,alte_heimat,,,,probable,1935,,,29min 04,,,probable 1935 29min 04,stern_IS_S_00142,Josef Stern,[Abgang von der Schule]
5,Wohnort,residence,"Ghettohaus, Walltorstraße 48",Ghettohaus,alte_heimat,,,,probable,ab 1935,,,,,https://www.giessen.de/index.php?ModID=7&FID=2...,probable ab 1935 https://www.giessen.de/index....,stern_IS_S_00142,Josef Stern,[Wohnort]
6,Hachschara,education,Schlesien,,alte_heimat,,,,probable,1935,,,30min 57s,,,probable 1935 30min 57s,stern_IS_S_00142,Josef Stern,[Hachschara]
7,Bei Verwandten,residence,Berlin,,alte_heimat,Q64,2950159.0,,probable,1935,,,31min 31s,,https://www.wikidata.org/wiki/Q64\nhttps://www...,2950159.0 probable 1935 31min 31s https://www....,stern_IS_S_00142,Josef Stern,"[Verwandte, Bei n]"
8,Jeschiwa,education,"Synagoge am Börneplatz, Frankfurt/M",Synagoge,alte_heimat,Q111620976,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q111620976\nhttp...,31min 58s — [Hinweis: Zerstört im Novemberpogr...,stern_IS_S_00142,Josef Stern,[Jeschiwa]
9,Abfahrt > Abschied von den Eltern,farewell,"Bahnhof, Frankfurt am Main",Bahnhof,alte_heimat,Q1794,6553153.0,,probable,1936,,,,,https://www.wikidata.org/wiki/Q1794\nhttps://w...,6553153.0 probable 1936 https://www.wikidata.o...,stern_IS_S_00142,Josef Stern,"[Abschied von den Eltern, Abfahrt, >]"


In [ ]:
import os
import re
import requests
import openpyxl

BASE_URL = os.environ.get("MMT_API_URL", "http://localhost:8000/motm/api")
LOGIN_URL = os.environ.get("MMT_LOGIN_URL", "http://localhost:8000/admin/login/")
SESSION = requests.Session()

SESSION.get(LOGIN_URL)
SESSION.post(LOGIN_URL, data={
    "username": "admin",
    "password": "admin",
    "csrfmiddlewaretoken": SESSION.cookies["csrftoken"],
    "next": "/admin/",
})
SESSION.headers["X-CSRFToken"] = SESSION.cookies.get("csrftoken", "")
SESSION.headers["Referer"] = LOGIN_URL
print("Authenticated" if SESSION.cookies.get("sessionid") else "Login failed")


def api_get(endpoint, params=None):
    if params is None:
        params = {}
    params.setdefault("limit", 10000)
    resp = SESSION.get(f"{BASE_URL}/{endpoint}/", params=params)
    resp.raise_for_status()
    data = resp.json()
    return data["results"] if isinstance(data, dict) and "results" in data else data


def api_post(endpoint, payload):
    resp = SESSION.post(f"{BASE_URL}/{endpoint}/", json=payload)
    resp.raise_for_status()
    return resp.json()


def api_patch(endpoint, obj_id, payload):
    resp = SESSION.patch(f"{BASE_URL}/{endpoint}/{obj_id}/", json=payload)
    resp.raise_for_status()
    return resp.json()


# --- Concepts ---
_concept_cache = {}


def get_or_create_concept(label):
    label = label.strip()
    if not label or label == "nan":
        return None
    key = label.lower()
    if key in _concept_cache:
        return _concept_cache[key]
    existing = api_get("concepts", params={"search": label})
    match = next((c for c in existing if c["label"].lower() == key), None)
    if match:
        _concept_cache[key] = match["id"]
        return match["id"]
    created = api_post("concepts", {"label": label})
    _concept_cache[key] = created["id"]
    return created["id"]


# --- Locations database (locations.xlsx) ---
LOCATIONS_XLSX = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".", "locations.xlsx")
_locations_db = {}  # _normalize_location(name) -> {lat, long, wikidata_id, geonames_id, label, urls}

def _extract_id_from_url(url, prefix):
    """Extract ID from a URL like https://www.wikidata.org/wiki/Q1794 -> Q1794"""
    if not url:
        return None
    url = str(url).strip()
    if not url or url == "nan":
        return None
    idx = url.rfind("/")
    return url[idx+1:] if idx >= 0 else url

def load_locations_db():
    """Load locations.xlsx into _locations_db."""
    if not os.path.exists(LOCATIONS_XLSX):
        print(f"  locations.xlsx not found at {LOCATIONS_XLSX}, starting fresh")
        return
    wb = openpyxl.load_workbook(LOCATIONS_XLSX)
    ws = wb.active
    for r in range(2, ws.max_row + 1):
        name = ws.cell(r, 1).value
        if not name:
            continue
        key = _normalize_location(name)
        wikidata_url = str(ws.cell(r, 6).value or "").strip()
        geonames_url = str(ws.cell(r, 5).value or "").strip()
        extra_url = str(ws.cell(r, 7).value or "").strip()
        _locations_db[key] = {
            "name": str(name).strip(),
            "lat": ws.cell(r, 2).value,
            "long": ws.cell(r, 3).value,
            "label": str(ws.cell(r, 4).value or "").strip(),
            "wikidata_id": _extract_id_from_url(wikidata_url, "wikidata.org"),
            "geonames_id": _extract_id_from_url(geonames_url, "geonames.org"),
            "wikidata_url": wikidata_url if wikidata_url and wikidata_url != "nan" else "",
            "geonames_url": geonames_url if geonames_url and geonames_url != "nan" else "",
            "extra_url": extra_url if extra_url and extra_url != "nan" else "",
        }
    print(f"  Loaded {len(_locations_db)} locations from locations.xlsx")

def _is_empty(value):
    """Check if a cell value is effectively empty."""
    return value is None or str(value).strip() in ("", "nan", "None")

def _normalize_location(name):
    """Normalize a location name for matching: lowercase, no punctuation, sorted words."""
    name = str(name).strip().lower()
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(sorted(name.split()))

def save_locations_db():
    """Save _locations_db back to locations.xlsx, merging with existing data."""
    HEADERS = ["location", "lat", "long", "label", "www.geonames.org", "www.wikidata.org", "www.giessen.de"]
    FIELD_COL = {1: "name", 2: "lat", 3: "long", 4: "label", 5: "geonames_url", 6: "wikidata_url", 7: "extra_url"}

    if os.path.exists(LOCATIONS_XLSX):
        wb = openpyxl.load_workbook(LOCATIONS_XLSX)
        ws = wb.active
        # Index existing rows by lowercase location name
        existing_rows = {}
        for r in range(2, ws.max_row + 1):
            name = ws.cell(r, 1).value
            if name:
                existing_rows[_normalize_location(name)] = r
        added = 0
        for key in sorted(_locations_db.keys()):
            loc = _locations_db[key]
            if key in existing_rows:
                row_num = existing_rows[key]
                for col_idx, field in FIELD_COL.items():
                    if col_idx == 1:
                        continue  # don't touch location name
                    if _is_empty(ws.cell(row_num, col_idx).value):
                        new_val = loc.get(field)
                        if not _is_empty(new_val):
                            ws.cell(row_num, col_idx).value = new_val
            else:
                ws.append([
                    loc["name"],
                    loc.get("lat") or "",
                    loc.get("long") or "",
                    loc.get("label", ""),
                    loc.get("geonames_url", ""),
                    loc.get("wikidata_url", ""),
                    loc.get("extra_url", ""),
                ])
                added += 1
    else:
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.append(HEADERS)
        added = 0
        for key in sorted(_locations_db.keys()):
            loc = _locations_db[key]
            ws.append([
                loc["name"],
                loc.get("lat") or "",
                loc.get("long") or "",
                loc.get("label", ""),
                loc.get("geonames_url", ""),
                loc.get("wikidata_url", ""),
                loc.get("extra_url", ""),
            ])
            added += 1
    wb.save(LOCATIONS_XLSX)
    print(f"  Saved locations.xlsx: {added} new rows added, {len(_locations_db)} total in db")

def upsert_location_db(name, wikidata_qid=None, geonames_id=None):
    """Add location to the database if not already present."""
    if not name or str(name).strip() in ("", "nan"):
        return
    name = str(name).strip()
    key = _normalize_location(name)
    if key in _locations_db:
        # Enrich existing entry with external IDs if provided
        if wikidata_qid and wikidata_qid not in ("", "nan") and not _locations_db[key].get("wikidata_id"):
            _locations_db[key]["wikidata_id"] = wikidata_qid
            _locations_db[key]["wikidata_url"] = f"https://www.wikidata.org/wiki/{wikidata_qid}"
        if geonames_id and geonames_id not in ("", "nan") and not _locations_db[key].get("geonames_id"):
            _locations_db[key]["geonames_id"] = geonames_id
            _locations_db[key]["geonames_url"] = f"https://www.geonames.org/{geonames_id}"
        return
    _locations_db[key] = {
        "name": name,
        "lat": None,
        "long": None,
        "label": "",
        "wikidata_id": wikidata_qid if wikidata_qid and wikidata_qid not in ("", "nan") else "",
        "geonames_id": geonames_id if geonames_id and geonames_id not in ("", "nan") else "",
        "wikidata_url": f"https://www.wikidata.org/wiki/{wikidata_qid}" if wikidata_qid and wikidata_qid not in ("", "nan") else "",
        "geonames_url": f"https://www.geonames.org/{geonames_id}" if geonames_id and geonames_id not in ("", "nan") else "",
        "extra_url": "",
    }

load_locations_db()

_location_cache = {}

def get_or_create_location(name, wikidata_qid=None, geonames_id=None):
    """Create or retrieve a LocationPoint, using locations.xlsx for coordinates."""
    if not name or str(name).strip() in ("", "nan"):
        return None
    name = str(name).strip()
    key = _normalize_location(name)
    if key in _location_cache:
        return _location_cache[key]
    existing = api_get("locations", params={"search": name})
    match = next((loc for loc in existing if _normalize_location(loc["current_name"]) == key), None)
    if match:
        _location_cache[key] = match["id"]
        return match["id"]
    # Build payload from locations.xlsx data
    db_entry = _locations_db.get(key, {})
    payload = {"current_name": name}
    # Coordinates from locations.xlsx
    lat = db_entry.get("lat")
    lng = db_entry.get("long")
    if lat and lng:
        try:
            payload["latitude"] = float(lat)
            payload["longitude"] = float(lng)
        except (ValueError, TypeError):
            pass
    # External IDs: prefer locations.xlsx, fall back to function args
    wid = db_entry.get("wikidata_id") or (wikidata_qid if wikidata_qid and wikidata_qid not in ("", "nan") else None)
    gid = db_entry.get("geonames_id") or (geonames_id if geonames_id and geonames_id not in ("", "nan") else None)
    if wid:
        payload["wikidata_id"] = wid
    if gid:
        payload["geonames_id"] = gid
    created = api_post("locations", payload)
    _location_cache[key] = created["id"]
    return created["id"]


# --- Persons ---
_person_cache = {}


def get_person_id(protagonist_dir, sheet_name):
    """Look up person by identifier extracted from directory name, then by sheet name."""
    if protagonist_dir in _person_cache:
        return _person_cache[protagonist_dir]
    # Try identifier: "stern_IS_S_00142" → "IS_S_00142"
    parts = protagonist_dir.split("_", 1)
    if len(parts) > 1:
        identifier = parts[1]
        existing = api_get("persons", params={"search": identifier})
        match = next((p for p in existing if p.get("identifier") == identifier), None)
        if match:
            _person_cache[protagonist_dir] = match["id"]
            return match["id"]
    # Fallback: search by sheet name (e.g. "Josef Stern")
    existing = api_get("persons", params={"search": sheet_name})
    if existing:
        _person_cache[protagonist_dir] = existing[0]["id"]
        return existing[0]["id"]
    return None


# --- Timespans ---
_timespan_cache = {}


def get_or_create_timespan(start=None, end=None):
    if start is None and end is None:
        return None
    s_iso = start.isoformat() if start else None
    e_iso = end.isoformat() if end else None
    key = (s_iso, e_iso)
    if key in _timespan_cache:
        return _timespan_cache[key]
    existing = api_get("timespans")
    for ts in existing:
        if ts.get("start") == s_iso and ts.get("end") == e_iso:
            _timespan_cache[key] = ts["id"]
            return ts["id"]
    created = api_post("timespans", {"start": s_iso, "end": e_iso})
    _timespan_cache[key] = created["id"]
    return created["id"]


# --- URLs ---
_url_cache = {}


def get_or_create_url(url_str):
    url_str = url_str.strip()
    if not url_str or url_str == "nan":
        return None
    if url_str in _url_cache:
        return _url_cache[url_str]
    existing = api_get("urls")
    for u in existing:
        if u["url"] == url_str:
            _url_cache[url_str] = u["id"]
            return u["id"]
    try:
        created = api_post("urls", {"url": url_str})
        _url_cache[url_str] = created["id"]
        return created["id"]
    except Exception as e:
        print(f"  URL creation failed for {url_str}: {e}")
        return None


def extract_urls_from_text(text):
    """Extract URLs from a text field."""
    if not text or str(text).strip() in ("", "nan"):
        return []
    return re.findall(r"https?://[^\s<>\"{}|\\^`\[\]]+", str(text))


def clean_str(val):
    """Return cleaned string or empty string for nan/blank values."""
    s = str(val).strip()
    if s in ("", "nan"):
        return ""
    # Strip .0 suffix from float-converted IDs (e.g. '2920512.0' -> '2920512')
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s


# === Main import ===

# 1. Create all concepts
print("Creating concepts...")
for label in sorted(concept_labels):
    get_or_create_concept(label)
print(f"  {len(concept_labels)} concept labels processed")

# 1b. Build concept taxonomy (parent + icon)
print("Setting up concept taxonomy...")
import json as _json
with open("taxonomy.json", "r") as _f:
    _tax = _json.load(_f)

# Create root categories with icons
_cat_ids = {}  # category key -> concept id
for key, info in _tax["categories"].items():
    cid = get_or_create_concept(info["label"])
    if cid:
        api_patch("concepts", cid, {"icon": info["icon"], "parent": None})
        _cat_ids[key] = cid

# Create sub-categories with parent
_subcat_ids = {}
for key, info in _tax["sub_categories"].items():
    cid = get_or_create_concept(info["label"])
    parent_id = _cat_ids.get(info["parent"])
    if cid and parent_id:
        root_icon = _tax["categories"][info["parent"]]["icon"]
        api_patch("concepts", cid, {"icon": root_icon, "parent": parent_id})
        _subcat_ids[key] = cid

# Set parent + icon on each mapped leaf concept
for label, cat_key in _tax["concept_to_category"].items():
    cid = _concept_cache.get(label.lower().strip())
    if not cid:
        cid = get_or_create_concept(label)
    if cid:
        root_icon = _tax["categories"][cat_key]["icon"]
        parent_id = _cat_ids.get(cat_key)
        api_patch("concepts", cid, {"icon": root_icon, "parent": parent_id})

# Set icon on unmapped concepts (default to empty)
print(f"  Taxonomy set up: {len(_cat_ids)} roots, {len(_subcat_ids)} sub-cats")


# 2. Import events
print("\nImporting events...")
created_events = []
errors = []

for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc="Events")):
    try:
        # Timespan from date_label
        date_label = clean_str(row["date_label"])
        ts = parse_timespan(date_label) if date_label else None
        timespan_id = get_or_create_timespan(ts.start, ts.end) if ts and ts.start else None

        # Upsert locations into the locations database
        upsert_location_db(
            row["end_location"],
            wikidata_qid=clean_str(row.get("wikidata_qid", "")),
            geonames_id=clean_str(row.get("geonames_id", "")),
        )
        upsert_location_db(row["start_location"])

        # Locations with external IDs
        end_loc_id = get_or_create_location(
            row["end_location"],
            wikidata_qid=clean_str(row.get("wikidata_qid", "")),
            geonames_id=clean_str(row.get("geonames_id", "")),
        )
        start_loc_id = get_or_create_location(row["start_location"])

        # Person
        person_id = get_person_id(row["protagonist"], row["name"])

        # is_confirmed from date_certainty
        date_certainty = clean_str(row.get("date_certainty", ""))
        is_confirmed = date_certainty.lower() in ("certain", "sicher", "yes", "ja")

        # Event description
        description = clean_str(row["event_label"])

        # Build event payload
        payload = {
            "description": description,
            "is_confirmed": is_confirmed,
        }
        if timespan_id:
            payload["timespan"] = timespan_id
        if start_loc_id:
            payload["start_location"] = start_loc_id
        if end_loc_id:
            payload["end_location"] = end_loc_id
        if person_id:
            payload["persons"] = [person_id]

        # URLs from external_links
        url_ids = []
        for url_str in extract_urls_from_text(row.get("external_links", "")):
            if "geonames.org" in url_str or "wikidata.org" in url_str:
                continue
            uid = get_or_create_url(url_str)
            if uid:
                url_ids.append(uid)
        if url_ids:
            payload["urls"] = url_ids

        # Concepts from event_type, place_type, place_category
        concept_ids = []
        for col in ["event_type", "place_type", "place_category"]:
            val = clean_str(row.get(col, ""))
            if val:
                cid = get_or_create_concept(val)
                if cid:
                    concept_ids.append(cid)

        if concept_ids:
            payload["concepts"] = concept_ids

        event = api_post("events", payload)
        event_id = event["id"]

        # Extraction to link concepts, source data, and notes
        source_quote = clean_str(row.get("source_quote", ""))
        source_timecode = clean_str(row.get("source_timecode", ""))
        source_doc = clean_str(row.get("source_doc", ""))
        memorial = clean_str(row.get("memorial_inscription", ""))

        # Build extraction notes from memorial_inscription, source_doc, original notes
        notes_parts = []
        if memorial:
            notes_parts.append(f"Memorial inscription: {memorial}")
        if source_doc:
            notes_parts.append(f"Source: {source_doc}")
        extraction_notes = "\n".join(notes_parts)

        if concept_ids or source_quote or extraction_notes:
            extraction_payload = {
                "event": event_id,
                "quote": source_quote,
                "timecode": source_timecode,
                "notes": extraction_notes,
            }
            if person_id:
                extraction_payload["people_mentioned"] = person_id
            if concept_ids:
                extraction_payload["concepts"] = concept_ids
            api_post("extractions", extraction_payload)

        created_events.append(event_id)
    except Exception as e:
        errors.append(f"Row {idx}: {e}")

print(f"\nCreated {len(created_events)} events")
if errors:
    print(f"\n{len(errors)} errors:")
    for err in errors[:20]:
        print(f"  {err}")

# Save updated locations database
save_locations_db()
